Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import KMeans
import networkx as nx

Load dataset with column names

In [3]:
columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
df = pd.read_csv('iris.data', header=None, names=columns)
df = df.dropna()
X = df.drop('class', axis=1).values

Compute distance matrix

In [4]:
dist_matrix = pairwise_distances(X, metric='euclidean')

Build graph

In [5]:
n = X.shape[0]
G = nx.Graph()

for i in range(n):
    for j in range(i + 1, n):
        G.add_edge(i, j, weight=dist_matrix[i][j])

Construct MST

In [6]:
mst = nx.minimum_spanning_tree(G, algorithm='kruskal')

Remove Largest Edges to Form Clusters

In [7]:
k = 3
edges = sorted(mst.edges(data=True), key=lambda x: x[2]['weight'], reverse=True)

for edge in edges[:k-1]:
    mst.remove_edge(edge[0], edge[1])

Extract Cluster Labels from MST

In [8]:
clusters = list(nx.connected_components(mst))
mst_labels = np.zeros(n)

for i, comp in enumerate(clusters):
    for node in comp:
        mst_labels[node] = i

Apply K-Means Clustering

In [9]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)

Save MST Clusters

In [10]:
pd.DataFrame({
    'SampleId': range(len(mst_labels)),
    'ClusterLabel': mst_labels.astype(int)
}).to_csv('mst_clusters.csv', index=False)

Save K-Means Clusters

In [11]:
pd.DataFrame({
    'SampleId': range(len(kmeans_labels)),
    'ClusterLabel': kmeans_labels
}).to_csv('kmeans_clusters.csv', index=False)

Compute Silhouette Scores

In [12]:
mst_score = silhouette_score(X, mst_labels)
kmeans_score = silhouette_score(X, kmeans_labels)

mst_score, kmeans_score

(np.float64(0.5118387098922373), np.float64(0.5525919445499757))

#Conclusion
K-Means clustering achieved a higher silhouette score compared to MST-based clustering, indicating better-defined and more compact clusters. This is expected because the Iris dataset contains relatively well-separated and approximately spherical clusters, which align well with the assumptions of K-Means. In contrast, MST-based clustering is more flexible and does not assume any specific cluster shape, but it is sensitive to edge removal and may split clusters incorrectly. As a result, MST did not perform as well as K-Means on this dataset. However, MST-based clustering is more suitable for datasets with non-spherical or irregularly shaped clusters.

MST-based clustering represents data points as nodes in a graph, where edges are weighted by pairwise distances. The Minimum Spanning Tree connects all points with the minimum total edge weight without forming cycles. By removing the largest edges, the tree is divided into clusters based on natural separations in the data. Since it does not rely on centroid-based assumptions, it can effectively capture clusters of arbitrary shapes. However, it is sensitive to noise and outliers, which may affect clustering quality.